# The sample

A miniature of AI-GenBench: **144 authentic + 144 fake** images, exactly 50/50, with 4 images from
each of the 36 benchmark generators.

It is partitioned into a **train** and a **val** half, 72 + 72 each way, stratified so every
generator contributes 2 images to each side and the three authentic sources keep their proportions.
A per-family score model cannot be calibrated on a family that landed entirely on one side, which is
what the stratification buys.

Building it is a one-time job and lives in [`imports/sample/`](imports/sample/) —
`authentic.py` pulls COCO / LAION / RAISE, `fakes.py` pulls the synthetic half from the benchmark's
parquet, and both write the `split` column through the same `common.assign_splits`. This notebook
just reads what they produced.

See the README for the caveats that matter before drawing conclusions from it — the important one
being that the two halves are **not content-paired**.

## Load

Read the two sets of images (real and fake) from disk into memory.

In [ ]:
import sys

sys.path.insert(0, "imports/sample")

import pandas as pd

import common
import imaging

REBUILD = False  # flip to True to re-import even when data/ is already populated

In [ ]:
if REBUILD or not common.sample_exists():
    import authentic
    import fakes

    authentic.main()
    print()
    fakes.main()
else:
    print("sample already imported — set REBUILD = True to rebuild")

sample = common.load_sample()
print(f"\n{len(sample)} rows")

## Balance

Count how many real vs. fake images to confirm the 144/144 split, and check the same balance holds
inside each of train and val.

In [ ]:
counts = sample["label"].value_counts().sort_index()
print(f"authentic (label 0): {counts.get(0, 0)}")
print(f"fake      (label 1): {counts.get(1, 0)}")
assert counts.get(0, 0) == counts.get(1, 0), "halves are not balanced"

by_split = pd.crosstab(sample["split"], sample["label"])
assert (by_split == len(sample) // 4).all().all(), f"train/val is not 50/50 within each class:\n{by_split}"
print()
print(by_split.to_string())

print()
print(sample.groupby(["label", "origin_dataset"]).size().to_string())

## Splits

The train/val partition. Two things have to hold: no image is on both sides, and no generator is
missing from either side — a generator absent from `train` has no score model, one absent from `val`
has nothing to check it against.

In [ ]:
train_ids = set(sample.loc[sample["split"] == "train", "file_id"])
val_ids = set(sample.loc[sample["split"] == "val", "file_id"])
assert not train_ids & val_ids, f"{len(train_ids & val_ids)} file_ids appear in both splits"
print(f"{len(train_ids)} train + {len(val_ids)} val, disjoint")

per_split = sample[sample["label"] == 1].groupby(["split", "generator"]).size()
assert (per_split == 2).all(), f"uneven generator coverage:\n{per_split[per_split != 2]}"
print(f"all {per_split.index.get_level_values('generator').nunique()} generators: "
      f"2 train + 2 val")

print()
print(pd.crosstab(sample[sample["label"] == 0]["origin_dataset"], sample["split"]).to_string())

## Generator coverage

Every generator should appear exactly four times. `release_date` is the benchmark's own registry —
the spine of the G≤T construction — and it stops at 2024-08, which is the coverage gap newer
generators would have to fill.

In [ ]:
fakes_only = sample[sample["label"] == 1]
per_generator = fakes_only.groupby(["release_date", "generator"]).size()

print(f"{len(per_generator)} generators, {per_generator.min()}-{per_generator.max()} images each")
print(f"{fakes_only['release_date'].min()} to {fakes_only['release_date'].max()}")
print()
print(per_generator.to_string())

## Compression parity

Check that the two halves are compressed with the same JPEG compression settings.

Both halves are normalized by the same `prepare_image`, so they must share one quantization table.
If they ever diverge, a detector could separate the classes on compression history alone.

In [ ]:
imaging.check_parity(common.AUTHENTIC_DIR, common.FAKES_DIR)

print()
print("fake source containers, before normalizing:")
print(fakes_only["source_format"].value_counts().to_string())

## Preview

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

shown = (
    sample[sample["label"] == 0].head(4).to_dict("records")
    + fakes_only.head(4).to_dict("records")
)

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for ax, row in zip(axes.flat, shown):
    ax.imshow(Image.open(common.PROJECT_ROOT / row["path"]))
    kind = "real" if row["label"] == 0 else f'fake · {row["generator"]}'
    ax.set_title(f'{kind}\n{row["origin_dataset"]} · {row["width"]}x{row["height"]}', fontsize=8)
for ax in axes.flat:
    ax.axis("off")
plt.tight_layout()

## The manifest

In [ ]:
sample[["file_id", "label", "generator", "origin_dataset", "split"]].head(10)